# Predict Customer Churn — End-to-End ML Pipeline

## Pipeline Overview

The notebook is structured as an **OOP pipeline** with 7 classes, each responsible for a single stage:

| Class | Responsibility |
|---|---|
| `DataLoader` | Load CSVs and validate columns |
| `DataPreprocessor` | Fix types, fill missing values |
| `FeatureEngineer` | Create 10 new features + encode categoricals |
| `ModelTrainer` | Train 5 models with 5-fold cross-validation |
| `ModelEvaluator` | Compare models and pick the best one |
| `SubmissionGenerator` | Train on full data and write `submission.csv` |
| `Pipeline` | Orchestrate all stages end-to-end |




In [14]:
# %%
import os
import warnings

warnings.filterwarnings("ignore")

# Use raw strings (r"...") for Windows paths
TRAIN_PATH = r"C:\projects\OOPs\myoops Solution NoteBook\Dataset\train.csv"
TEST_PATH = r"C:\projects\OOPs\myoops Solution NoteBook\Dataset\test.csv"

OUTPUT_PATH = r"C:\projects\OOPs\myoops Solution NoteBook\Dataset\sample_submission.csv"


print("Checking dataset files...\n")

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"Training file not found:\n{TRAIN_PATH}\n\n"
        "Check your project folder structure."
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"Test file not found:\n{TEST_PATH}\n\n"
        "Check your project folder structure."
    )

print("Training file found  ✓")
print("Test file found      ✓")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier


from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 8. GLOBAL CONSTANTS

RANDOM_STATE = 42
N_SPLITS = 5



print("\nAll imports loaded successfully ✓")
print(f"Train path : {TRAIN_PATH}")
print(f"Test path  : {TEST_PATH}")
print(f"Random seed: {RANDOM_STATE}")
print(f"CV folds   : {N_SPLITS}")

Checking dataset files...

Training file found  ✓
Test file found      ✓

All imports loaded successfully ✓
Train path : C:\projects\OOPs\myoops Solution NoteBook\Dataset\train.csv
Test path  : C:\projects\OOPs\myoops Solution NoteBook\Dataset\test.csv
Random seed: 42
CV folds   : 5


## Stage 1 — Loading Data


In [42]:
class DataLoader:
    """
    Loads train and test CSVs and performs basic sanity checks.

    Why a class instead of a function?
    - The loaded DataFrames are stored as instance attributes (self.train, self.test),
      so downstream stages can inspect them if needed without re-reading from disk.
    """
    
    def __init__(self, train_path: str, test_path: str):
        self.train_path = train_path
        self.test_path = test_path
        self.train = None
        self.test = None
        
    def loader(self) -> tuple:
        self.train = pd.read_csv(self.train_path)
        self.test = pd.read_csv(self.test_path)
        
        print("\nTrain Data - First 5 Rows:")
        print(self.train.head())
        
        print(f'train shape : {self.train.shape}')
        print(f'test shape : {self.test.shape}')
        
        
        
        # Class Distribution
        churn_counts = self.train['Churn'].value_counts()
        churn_rate = self.train['Churn'].value_counts(normalize = True)
        
        
        
        print(f'\nchurn distribution : \n {churn_counts}')
        print(f'\nChurn rate (Yes): {churn_rate["Yes"]:.4f}  →  class imbalance ratio ≈ {churn_counts["No"]/churn_counts["Yes"]:.2f}:1')

        
        return self.train, self.test
    
print('Dataloader ready')        

Dataloader ready


In [43]:
# %%
data_loader = DataLoader(TRAIN_PATH, TEST_PATH)

train_df = data_loader.loader()


Train Data - First 5 Rows:
   id  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0   0    Male              0     Yes        Yes      29          Yes   
1   1    Male              0     Yes        Yes      58          Yes   
2   2    Male              0     Yes         No      58          Yes   
3   3  Female              0      No         No       1          Yes   
4   4  Female              0      No         No       1          Yes   

  MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0            No             DSL            Yes  ...              Yes   
1            No             DSL            Yes  ...               No   
2           Yes     Fiber optic             No  ...               No   
3            No     Fiber optic             No  ...               No   
4            No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0         Yes          No   